# 🔗 Cross-Domain Data Product — Federated Consumption
## Catalog: `data_mesh_hub` | Owner: Data Platform Team

This notebook demonstrates the **core Data Mesh pattern**: a **cross-domain data product** that consumes from multiple domain catalogs via **cross-catalog JOINs**.

**Key Data Mesh Principles Demonstrated:**
- ✅ **Reuse Across Teams** — ADOIT (EA) + SNOW (ITSM) gold products joined
- ✅ **Federated Governance** — Each domain retains ownership; hub only reads
- ✅ **Cross-Catalog Lineage** — UC tracks lineage across `adoit_product` → `data_mesh_hub`
- ✅ **Composite Business Value** — Neither domain alone can produce the risk profile

## Why This Matters
```
 adoit_product.gold                snow_product.gold              data_mesh_hub.cross_domain
┌──────────────────────┐    ┌────────────────────┐    ┌──────────────────────┐
│ application_landscape │───►│   service_health    │───►│ application_risk_    │
│ (EA Team owns)        │    │ (ITSM Team owns)   │    │   profile             │
│                       │    │                     │    │ (Platform Team owns)  │
│ • app metadata         │    │ • incident metrics  │    │ • combined risk view   │
│ • capabilities          │    │ • change metrics    │    │ • composite scoring    │
│ • criticality           │    │ • SLA compliance    │    │ • risk factors         │
└──────────────────────┘    └────────────────────┘    └──────────────────────┘
```
The **Application Risk Profile** cannot be built by either domain alone — it requires combining EA context (criticality, lifecycle) with operational reality (incidents, SLA breaches).

## 📦 Source Data Products
Let's first inspect what each domain's gold product provides.

In [0]:
%sql
-- From adoit_product catalog (EA Team's domain)
SELECT app_id, app_name, criticality, lifecycle_status, 
       is_cloud_hosted, capability_count, quality_score
FROM adoit_product.gold.application_landscape
ORDER BY criticality;

In [0]:
%sql
-- From snow_product catalog (ITSM Team's domain)
SELECT affected_application_id, total_incidents, open_incidents, 
       critical_incidents, sla_breaches, sla_compliance_pct, risk_score
FROM snow_product.gold.service_health
ORDER BY risk_score;

## 🔗 Cross-Catalog JOIN — The Data Mesh in Action
This is the key operation: joining data from `adoit_product` catalog with data from `snow_product` catalog to produce a federated product in `data_mesh_hub`.

Unity Catalog makes this seamless — no data movement, no ETL, just a cross-catalog SQL JOIN.

In [0]:
%sql
-- ============================================================
-- CROSS-CATALOG JOIN: adoit_product + snow_product → data_mesh_hub
-- This is the core Data Mesh federated consumption pattern
-- ============================================================
MERGE INTO data_mesh_hub.cross_domain.application_risk_profile AS target
USING (
    SELECT 
        al.app_id, al.app_name, al.app_owner, al.department,
        al.criticality, al.lifecycle_status, al.technology_stack,
        al.is_cloud_hosted, al.app_age_years, al.capability_count,
        COALESCE(sh.total_incidents, 0) AS total_incidents,
        COALESCE(sh.open_incidents, 0) AS open_incidents,
        COALESCE(sh.critical_incidents, 0) AS critical_incidents,
        COALESCE(sh.sla_breaches, 0) AS sla_breaches,
        COALESCE(sh.sla_compliance_pct, 100.0) AS sla_compliance_pct,
        sh.avg_resolution_hours,
        COALESCE(sh.total_changes, 0) AS total_changes,
        COALESCE(sh.change_success_rate_pct, 100.0) AS change_success_rate_pct,
        COALESCE(sh.risk_score, 'Low') AS service_risk_score,
        -- Composite risk: business criticality + operational health
        CASE 
            WHEN al.criticality = 'Critical' AND COALESCE(sh.risk_score, 'Low') IN ('Critical', 'High') THEN 'CRITICAL'
            WHEN al.criticality = 'Critical' AND COALESCE(sh.sla_breaches, 0) > 0 THEN 'HIGH'
            WHEN al.criticality IN ('Critical', 'High') AND COALESCE(sh.open_incidents, 0) > 0 THEN 'HIGH'
            WHEN al.lifecycle_status = 'Retiring' THEN 'MEDIUM'
            WHEN COALESCE(sh.total_incidents, 0) > 2 THEN 'MEDIUM'
            ELSE 'LOW'
        END AS composite_risk_score,
        CONCAT_WS(' | ',
            CASE WHEN al.criticality = 'Critical' THEN 'Business-critical app' END,
            CASE WHEN al.lifecycle_status = 'Retiring' THEN 'Retiring' END,
            CASE WHEN COALESCE(sh.open_incidents, 0) > 0 THEN CONCAT(sh.open_incidents, ' open incident(s)') END,
            CASE WHEN COALESCE(sh.sla_breaches, 0) > 0 THEN CONCAT(sh.sla_breaches, ' SLA breach(es)') END,
            CASE WHEN COALESCE(sh.critical_incidents, 0) > 0 THEN CONCAT(sh.critical_incidents, ' P1 incident(s)') END,
            CASE WHEN al.app_age_years > 10 THEN 'Legacy (>10 yrs)' END
        ) AS risk_factors,
        current_timestamp() AS product_generated_at
    FROM adoit_product.gold.application_landscape al       -- EA Team's catalog
    LEFT JOIN snow_product.gold.service_health sh           -- ITSM Team's catalog
        ON al.app_id = sh.affected_application_id
) AS source
ON target.app_id = source.app_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
-- The cross-domain data product: Application Risk Profile
SELECT app_id, app_name, criticality, lifecycle_status,
       total_incidents, open_incidents, sla_breaches, sla_compliance_pct,
       composite_risk_score, risk_factors
FROM data_mesh_hub.cross_domain.application_risk_profile
ORDER BY 
    CASE composite_risk_score WHEN 'CRITICAL' THEN 1 WHEN 'HIGH' THEN 2 WHEN 'MEDIUM' THEN 3 ELSE 4 END;

## 💼 Consumer Use Cases
Different teams consume the cross-domain product for different purposes:

In [0]:
%sql
-- Use Case 1: CTO Office — Which critical apps have the most risk?
SELECT app_name, criticality, composite_risk_score, 
       total_incidents, sla_breaches, risk_factors
FROM data_mesh_hub.cross_domain.application_risk_profile
WHERE composite_risk_score IN ('CRITICAL', 'HIGH')
ORDER BY composite_risk_score;

In [0]:
%sql
-- Use Case 2: Risk Committee — SLA compliance across the portfolio
SELECT 
    department,
    COUNT(*) AS app_count,
    SUM(total_incidents) AS total_incidents,
    SUM(sla_breaches) AS total_sla_breaches,
    ROUND(AVG(sla_compliance_pct), 1) AS avg_sla_compliance,
    SUM(CASE WHEN composite_risk_score IN ('CRITICAL', 'HIGH') THEN 1 ELSE 0 END) AS high_risk_apps
FROM data_mesh_hub.cross_domain.application_risk_profile
GROUP BY department
ORDER BY total_sla_breaches DESC;

In [0]:
%sql
-- Cross-domain product metadata — note the lineage referencing BOTH domain catalogs
SHOW TBLPROPERTIES data_mesh_hub.cross_domain.application_risk_profile;